In [8]:
if (!requireNamespace("mixOmics", quietly = TRUE)) {
    options(BioC_mirror = "https://bioconductor.org")
    BiocManager::install(version = "3.18", force = TRUE)
    BiocManager::install("mixOmics")
}

In [43]:
DATA_DIR <- "/home/mariya/Data/pallas"
TRAIN_DATA_DIR <- file.path(DATA_DIR, "nf-core-mlomix-train-bimodal/finalize")
TEST_DATA_DIR <- file.path(DATA_DIR, "nf-core-mlomix-test-bimodal/finalize")
METH_TRAIN_FILE <- file.path(TRAIN_DATA_DIR, "features.dnam.csv")
GEX_TRAIN_FILE <- file.path(TRAIN_DATA_DIR, "features.gex.csv")
LABELS_TRAIN_FILE <- file.path(TRAIN_DATA_DIR, "labels.csv")
METH_TEST_FILE <- file.path(TEST_DATA_DIR, "features.dnam.csv")
GEX_TEST_FILE <- file.path(TEST_DATA_DIR, "features.gex.csv")
LABELS_TEST_FILE <- file.path(TEST_DATA_DIR, "labels.csv")

In [44]:
library(mixOmics)
library(caret)

In [50]:
labels_train <- read.csv(LABELS_TRAIN_FILE, row.names = 1, check.names = FALSE, stringsAsFactors = FALSE)
labels_test <- read.csv(LABELS_TEST_FILE, row.names = 1, check.names = FALSE, stringsAsFactors = FALSE)
head(rownames(labels_train))
head(rownames(labels_test))

[1] "TARGET-20-PASWLN-D" "TARGET-20-PAVNHN-D" "TARGET-20-PARURW-D"
[4] "TARGET-20-PAPVKG-D" "TARGET-20-PASJDV-D" "TARGET-20-PASAFH-D"

[1] "TARGET-20-PAWCIL-D" "TARGET-20-PATAVF-D" "TARGET-20-PARNFZ-D"
[4] "TARGET-20-PAWWXU-D" "TARGET-20-PAVALK-D" "TARGET-20-PAWXCK-D"

In [51]:
meth_train <- read.csv(METH_TRAIN_FILE, row.names = 1, check.names = FALSE)
gex_train <- read.csv(GEX_TRAIN_FILE, row.names = 1, check.names = FALSE)
head(rownames(meth_train))
head(rownames(gex_train))
head(rownames(labels_train))

[1] "TARGET-20-PASWLN-D" "TARGET-20-PAVNHN-D" "TARGET-20-PARURW-D"
[4] "TARGET-20-PAPVKG-D" "TARGET-20-PASJDV-D" "TARGET-20-PASAFH-D"

[1] "TARGET-20-PASWLN-D" "TARGET-20-PAVNHN-D" "TARGET-20-PARURW-D"
[4] "TARGET-20-PAPVKG-D" "TARGET-20-PASJDV-D" "TARGET-20-PASAFH-D"

[1] "TARGET-20-PASWLN-D" "TARGET-20-PAVNHN-D" "TARGET-20-PARURW-D"
[4] "TARGET-20-PAPVKG-D" "TARGET-20-PASJDV-D" "TARGET-20-PASAFH-D"

In [46]:
meth_test <- read.csv(METH_TEST_FILE, row.names = 1, check.names = FALSE)
gex_test <- read.csv(GEX_TEST_FILE, row.names = 1, check.names = FALSE)
head(rownames(meth_test))
head(rownames(gex_test))
head(rownames(labels_test))


[1] "TARGET-20-PAWCIL-D" "TARGET-20-PATAVF-D" "TARGET-20-PARNFZ-D"
[4] "TARGET-20-PAWWXU-D" "TARGET-20-PAVALK-D" "TARGET-20-PAWXCK-D"

[1] "TARGET-20-PAWCIL-D" "TARGET-20-PATAVF-D" "TARGET-20-PARNFZ-D"
[4] "TARGET-20-PAWWXU-D" "TARGET-20-PAVALK-D" "TARGET-20-PAWXCK-D"

[1] "TARGET-20-PAWCIL-D" "TARGET-20-PATAVF-D" "TARGET-20-PARNFZ-D"
[4] "TARGET-20-PAWWXU-D" "TARGET-20-PAVALK-D" "TARGET-20-PAWXCK-D"

In [52]:
# ── 2. Align samples ──────────────────────────────────────────────────────────

align_samples <- function(meth, rna, labels) {
  common <- Reduce(intersect, list(rownames(meth), rownames(rna), rownames(labels)))
  cat("Common samples:", length(common), "\n")
  list(
    meth   = meth[common, ],
    rna    = rna[common, ],
    labels = labels[common, ]
  )
}

train <- align_samples(meth_train, gex_train, labels_train)
test  <- align_samples(meth_test,  gex_test,  labels_test)

Common samples: 843 
Common samples: 362 
